# 01 - Dataset audit (Stage 0)

**Question this notebook answers:** before any feature engineering or modelling, is the
ASVspoof 2019 LA split we are about to use *safe to model on* -- i.e. is the bonafide/spoof
label predictable from nuisance metadata alone (duration, sample rate, codec, loudness,
silence, source corpus, speaker identity), rather than from the speech signal itself?

**Mode:** this notebook checks for a real ASVspoof 2019 LA download at
`configs/data.yaml: dataset.root`. If absent (the expected state while the user's download is
in progress), it builds a small **synthetic mini-manifest** instead -- synthesised audio
written to a temp directory plus a fake protocol file in the exact ASVspoof 2019 LA format --
so every cell below still executes and is reviewable today. The active mode is printed by the
next cell and repeated at the end. **Any numeric verdict below is only meaningful in real
mode**; in synthetic mode the same code path is exercised on data manufactured to be roughly
class-balanced, so a "looks safe" verdict there is a pipeline check, not a finding about
ASVspoof.

Imports only from `audio_df` (`data`, `leakage`) -- no feature extraction, splitting, or
model code is defined in this notebook.

In [ ]:
from __future__ import annotations

import tempfile
import warnings
from pathlib import Path

import pandas as pd
import yaml

from audio_df import data, leakage
from audio_df.synthetic import build_synthetic_asvspoof_tree


def _find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(6):
        if (cur / "pyproject.toml").exists():
            return cur
        cur = cur.parent
    raise RuntimeError(f"could not find repo root (pyproject.toml) from {start}")


REPO_ROOT = _find_repo_root(Path.cwd())
pd.set_option("display.width", 120)

In [ ]:
# --- mode detection ---------------------------------------------------------------------
_data_cfg_path = REPO_ROOT / "configs" / "data.yaml"
_real_cfg = yaml.safe_load(_data_cfg_path.read_text())
_real_root = Path(_real_cfg["dataset"]["root"])
if not _real_root.is_absolute():
    _real_root = REPO_ROOT / _real_root

USE_REAL_DATA = _real_root.exists() and any(_real_root.rglob("*.flac"))

if USE_REAL_DATA:
    MODE = "REAL DATA"
    cfg = _real_cfg
    cfg["dataset"]["root"] = str(_real_root)
    cfg["dataset"]["protocol_dir"] = str(REPO_ROOT / cfg["dataset"]["protocol_dir"])
    _tmpdir_handle = None
else:
    MODE = "SYNTHETIC MINI-MANIFEST (no ASVspoof download found)"
    _tmpdir_handle = tempfile.TemporaryDirectory(prefix="asvspoof_synth_")
    cfg = build_synthetic_asvspoof_tree(Path(_tmpdir_handle.name), seed=1337)

print(f"MODE: {MODE}")
print(f"dataset.root = {cfg['dataset']['root']}")

In [ ]:
manifest = data.build_manifest(cfg)
print(f"manifest: {len(manifest)} rows, columns = {list(manifest.columns)}")
manifest.head()

## Manifest summary: class / attack / speaker counts

`data.summarise_manifest` groups by (split, label, attack_id) and reports utterance and
distinct-speaker counts per cell.

In [ ]:
summary = data.summarise_manifest(manifest)
summary

In [ ]:
class_counts = manifest.groupby(["split", "label"])["utt_id"].count().unstack(fill_value=0)
print("Class balance per split:")
class_counts

class is very imbalanced, it was designed this way to reflect the real-world prevalence of spoofing attacks. Therefore, metrics use: EER, minDCF, Per-attack EER. 
Accuracy is bad metric, F1 depends on the ratio

### Observation
The table above shows how many bonafide/spoof utterances (and distinct speakers per
attack/label cell) each split contains. In real mode, ASVspoof 2019 LA train/dev only ever
contain attacks **A01-A06**; eval contains **A07-A19**, which never appear in train/dev.

### Inference
Because train/dev and eval never share an attack family, the official eval split is already
an **unseen-attack generalisation test** by construction, not an optional extra we chose to
add. A model that scores well on dev (seen attacks A01-A06) but worse on eval (unseen attacks
A07-A19) is not necessarily overfitting the training distribution in the usual sense -- it may
simply be facing spoofing algorithms it has never seen a single example of. Any dev/eval gap
must be interpreted with this asymmetry in mind, not read as "the model overfit."

## Speaker-overlap check

ASVspoof protocols are speaker-disjoint across train/dev by design. `assert_no_speaker_overlap`
raises (naming the offending speakers) if that is violated -- which would be a serious leak,
since a model could then simply recognise a voice it saw labelled during training.

In [ ]:
train_df = manifest[manifest["split"] == "train"].reset_index(drop=True)
dev_df = manifest[manifest["split"] == "dev"].reset_index(drop=True)

try:
    data.assert_no_speaker_overlap(train_df, dev_df)
    print("PASS: no speaker overlap between train and dev.")
except ValueError as exc:
    print(f"FAIL: {exc}")

## Split-content hash

`split_hash` is a stable content hash (independent of row order) so two runs can prove they
used exactly the same split -- computed here twice to show it is deterministic.

In [ ]:
h1 = data.split_hash(train_df)
h2 = data.split_hash(train_df.sample(frac=1.0, random_state=0).reset_index(drop=True))
print(f"hash (original order):  {h1}")
print(f"hash (shuffled rows):   {h2}")
assert h1 == h2, "split_hash should be invariant to row order"
print("PASS: split_hash is stable under row reordering.")

## Leakage audit (Stage 0)

Fit tiny classifiers (logistic regression, depth-2 decision tree) on **metadata-only**
features -- duration, sample rate, codec, source corpus, speaker identity, RMS loudness, and
leading/trailing/total silence ratio -- via out-of-fold cross-validation, and report each
feature's AUC/EER at predicting bonafide/spoof alone. `leakage.py`'s `threshold=0.65` is a
**project-local convention, not a literature value** (see `leakage.py` module docstring / the
one directly comparable published number, Müller et al. 2021, arXiv:2106.12914, a single
leading-silence feature reaching 15.1% EER on ASVspoof 2019 LA eval -- well past this
threshold, so the convention is deliberately conservative-but-not-paranoid, not arbitrary).

In [ ]:
nuisance = leakage.extract_nuisance_features(manifest, audio_root=None)
nuisance.head()

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    audit_report = leakage.audit_all(nuisance, manifest["label"], seed=cfg.get("seed", 1337))

for w in caught:
    if issubclass(w.category, UserWarning):
        print(f"WARNING: {w.message}")

audit_report

### Observation
The table above reports, for every nuisance feature and for the combined-feature model, the
out-of-fold AUC/EER (with bootstrap 95% CIs) at predicting the bonafide/spoof label from
metadata alone, and whether that AUC exceeds the project's 0.65 flag threshold. Any `WARNING:`
line printed above names a feature that crossed that threshold.

### Inference
A flagged feature means a real detector trained on this split could learn to key off that
nuisance property (e.g. "spoof clips are a bit louder/quieter" or "this speaker's utterances
are always spoof") instead of learning anything about synthetic-speech acoustics -- exactly
the `"clean WAV -> fake"` failure mode the handoff warns about. In **synthetic mode**, this
mini-manifest was deliberately built with a per-attack gain change (to also serve as a
generator-conditioned spectral effect in notebooks 02/03), so if `rms` or a silence feature is
flagged here, that is an artifact of *how this synthetic corpus was generated*, not a finding
about ASVspoof -- see the mode banner. In **real mode**, any flagged feature here should be
treated as a genuine methodology concern and investigated (e.g. per-attack recording chain
differences) before proceeding to Stage 1/2 EDA.

## Class distributions of the audited nuisance features

Aggregate distributions (never a single cherry-picked example), one panel per feature, class
overlaid.

In [ ]:

import matplotlib.pyplot as plt

from audio_df import visualization as viz

plot_cols = [
    c for c in nuisance.columns
    if nuisance[c].notna().any() and nuisance[c].dtype.kind in "fi"
]
n = len(plot_cols)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 3.2))
if n == 1:
    axes = [axes]
for ax, col in zip(axes, plot_cols, strict=False):
    values_by_class = {
        label: nuisance.loc[manifest["label"] == label, col].dropna().to_numpy()
        for label in manifest["label"].unique()
    }
    viz.plot_class_distributions(values_by_class, ax=ax)
    ax.set_title(col)
fig.tight_layout()
plt.show()

### Observation
Each panel overlays the bonafide and spoof distribution of one numeric nuisance feature. Any
panel where the two class curves are clearly separated is a candidate leakage channel -- the
same information the AUC table above already quantifies, shown here as a distribution rather
than a single number.

### Inference
A separated distribution here is *consistent with* (not independent proof beyond) a flagged
AUC above; the numeric audit is the primary evidence, this plot is a sanity-check visualization
of the same claim.

## Verdict

Computed programmatically from the audit table above, then stated in words.

In [ ]:
flagged = audit_report[audit_report["flagged"]]
print(f"Mode: {MODE}")
print(f"Flagged features (AUC > 0.65): {list(flagged['feature']) if len(flagged) else 'none'}")
if len(flagged) == 0:
    print("VERDICT: no metadata-only shortcut was found by these tiny classifiers -- "
          "this split is SAFE to proceed to Stage 1/2 modelling from this audit's perspective.")
else:
    print("VERDICT: at least one nuisance feature predicts the label above the project's "
          "convention threshold -- this split should NOT be modelled on as-is until the "
          "flagged feature(s) are investigated (e.g. is it a per-attack recording-chain "
          "artifact, or genuinely inherent to the corpus?).")

This verdict only carries real evidentiary weight when `MODE = REAL DATA` above. In synthetic
mode it exercises the exact same code path the real download will run through, so the
notebook is fully reviewable today; re-running it once the ASVspoof 2019 LA download completes
will automatically switch to real mode and produce the real verdict, with no code changes.